# 04 · Chat UIs and dynamic context

`gr.ChatInterface` calls `fn(message, history)` on every turn. The callback has to rebuild the full
message list from that history each time, because the model is stateless.

Use case: a support assistant for a fictional hosting company, "Nimbus Cloud".
The system prompt acts as the product spec; extra context is injected only when it's relevant.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))  # repo root, for `toolkit`

In [ ]:
import gradio as gr

from toolkit import stream
from toolkit.chat_ui import accumulate, build_messages

PROVIDER = "ollama"

## History format

With `type="messages"`, history is a list of `{"role", "content", "metadata", "options"}` dicts.
Most providers reject unknown keys, so `build_messages` strips everything but `role` and `content`.

In [ ]:
SYSTEM = """You are the support assistant for Nimbus Cloud, a managed hosting provider.
Plans: Starter $9/mo (1 app, 1GB RAM), Team $49/mo (10 apps, 8GB, preview deploys),
Scale $199/mo (unlimited apps, autoscaling, 99.95% SLA).
Recommend the smallest plan that fits. If unsure, ask about their traffic and team size.
Never promise features that aren't listed. Keep answers under 80 words."""


def support(message, history):
    yield from accumulate(stream(build_messages(SYSTEM, history, message), provider=PROVIDER))


gr.ChatInterface(support, type="messages", title="Nimbus Cloud support").launch()

## Dynamic context injection

Rather than putting every policy into the system prompt (cost, dilution of attention), detect what the
user is asking about and append only the relevant policy. Here keyword matching does the detection.
This is the core idea behind RAG, where embeddings and a vector store do the retrieval instead.

In [ ]:
POLICIES = {
    ("refund", "money back", "cancel"): (
        "Refunds: full refund within 14 days of first payment; "
        "afterwards, cancel anytime with no refund for the current month."
    ),
    ("outage", "down", "incident", "sla"): (
        "Incidents: status at status.nimbus.example. "
        "Scale plan SLA credits are 10% per 0.1% below 99.95%."
    ),
    ("gpu", "ml", "model"): (
        "GPUs are not offered. Suggest a dedicated ML platform and offer to host the web app part."
    ),
}


def relevant_policies(message: str) -> list[str]:
    text = message.lower()
    return [policy for keywords, policy in POLICIES.items() if any(k in text for k in keywords)]


def support_with_context(message, history):
    system = SYSTEM
    if extra := relevant_policies(message):
        system += "\n\nRelevant policy:\n" + "\n".join(extra)
    yield from accumulate(stream(build_messages(system, history, message), provider=PROVIDER))


relevant_policies("Can I get my money back if the site is down a lot?")

In [ ]:
gr.ChatInterface(support_with_context, type="messages", title="Nimbus Cloud support").launch()

Limits of keyword matching: no synonyms ("reimburse"), no paraphrase handling, and it breaks down past
a handful of documents. That's the motivation for semantic retrieval later on.